In [ ]:
import numpy as np
import pandas as pd

In [ ]:
df = pd.read_csv('train.csv')

In [ ]:
df['Embarked'].unique()

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin

In [ ]:
from sklearn.impute import SimpleImputer
class ImputedData(BaseEstimator, TransformerMixin):
    
    def fit(self, X, y=None):
        return self

    def transform(self, X, y=None):
        imputer = SimpleImputer(strategy = 'mean')
        X["ImputerAge"] = imputer.fit_transform(X[['Age']])
        return X


In [ ]:
from sklearn.preprocessing import OneHotEncoder
class EncodedData(BaseEstimator, TransformerMixin):
    
    def fit(self, X, y=None):
        return self

    def transform(self, X, y=None):
        X['gender']=np.where(X['Sex'] == 'female',1,0)
        X = X.drop(columns=['Sex'])
        encoder = OneHotEncoder()
        matrix = encoder.fit_transform(X[['Embarked']]).toarray()
        column_name = ['C','S','Q','N']
        for i in range(len(matrix.T)):
            X[column_name[i]]= matrix.T[i]
        maskPclass1 = X['Pclass']==1
        maskPclass3 = X['Pclass']==3
        X.loc[maskPclass1,'Pclass']=3
        X.loc[maskPclass3,'Pclass']=1
        return X    

   

In [ ]:
class FeatureDropper(BaseEstimator, TransformerMixin):
    
    def fit(self, X, y=None):
        return self

    def transform(self, X, y=None):
        X=X.drop(['Sex','Embarked','Name','Age','Cabin','N'],axis=1,errors='ignore')
        return X

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
pipeline = Pipeline([('imputedData',ImputedData()),
                    ("encodedData",EncodedData()),
                    ('featureDropper',FeatureDropper()),
                    ('Scaler',StandardScaler())])

In [ ]:
from sklearn.model_selection import train_test_split
df.columns

In [ ]:
X= df[['Sex','Embarked','Age','Pclass']]
y= df['Survived'].values

In [ ]:
X_train,X_test,y_train,y_test = train_test_split(X,y,random_state=0,test_size=0.2,stratify=y)

In [ ]:
X_train_scaled = pipeline.fit_transform(X_train)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

In [ ]:
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}
random_forest = RandomForestClassifier()

In [ ]:
grid_search = GridSearchCV(random_forest, param_grid, cv=5, scoring='accuracy')

In [ ]:
grid_search.fit(X_train_scaled, y_train)

In [ ]:
print("Best Parameters: ", grid_search.best_params_)
print("Best Accuracy: ", grid_search.best_score_)

In [ ]:
X_test_scaled = pipeline.transform(X_test)

In [ ]:
grid_search.score(X_test_scaled,y_test)

In [ ]:
X_scaled = pipeline.fit_transform(X)

In [ ]:
grid_search.fit(X_scaled,y)

In [ ]:
print("Best Parameters: ", grid_search.best_params_)
print("Best Accuracy: ", grid_search.best_score_)

In [ ]:
estimator = grid_search.best_estimator_

In [ ]:
estimator

In [ ]:
test_df = pd.read_csv("test.csv")

In [ ]:
test_df_final = test_df[['Sex','Embarked','Age','Pclass']]

In [ ]:
test_df_final = pipeline.transform(test_df_final)

In [ ]:
test_df_final

In [ ]:
predictions = estimator.predict(test_df_final)

In [ ]:
final_df = pd.DataFrame(test_df['PassengerId'])

In [ ]:
final_df['Survived'] = predictions

In [ ]:
final_df.to_csv("predictions.csv",index=False)